In [1]:
import time
notebook_start = time.perf_counter()

import os
import numpy  as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import thermoift.PLOT_SETTINGS as ps
import feos

from thermoift.rng_utils                import get_rng
from thermoift.FeosPlugin               import (RegistryManager, ParameterBuilder, VLECalculator, CompositionHandler)
from sklearn.preprocessing              import StandardScaler
from sklearn.pipeline                   import Pipeline
from sklearn.gaussian_process           import GaussianProcessRegressor
from sklearn.gaussian_process.kernels   import Matern, ConstantKernel, WhiteKernel

RegistryManager.load_registry()

({'carbon dioxide': PureRecord(identifier: {cas: 124-38-9, name: carbon dioxide, iupac_name: carbon dioxide}, molarweight: 44.01, m: 1.6298, sigma: 3.0867, epsilon_k: 163.34, q: 3.9546),
  'hydrogen': PureRecord(identifier: {cas: 1333-74-0, name: hydrogen, iupac_name: molecular hydrogen}, molarweight: 2.016, m: 0.68, sigma: 3.54, epsilon_k: 31.57),
  'nitrogen': PureRecord(identifier: {cas: 7727-37-9, name: nitrogen, iupac_name: molecular nitrogen}, molarweight: 28.01, m: 1.1879, sigma: 3.3353, epsilon_k: 90.99, q: 1.1151, association_sites: [{nb: 2.0}]),
  'argon': PureRecord(identifier: {cas: 7440-37-1, name: argon, iupac_name: argon}, molarweight: 39.962, m: 1.0, sigma: 3.37751, epsilon_k: 117.80903),
  'methane': PureRecord(identifier: {cas: 74-82-8, name: methane, iupac_name: methane}, molarweight: 16.031, m: 1.0, sigma: 3.70051, epsilon_k: 150.07147),
  'oxygen': PureRecord(identifier: {cas: 7782-44-7, name: oxygen, iupac_name: molecular oxygen}, molarweight: 31.99, m: 1.14702, s

In [2]:
# ═══════════════════════════════════════════════════════════════════════════════
# CONFIG  — AL_ST single-temperature multi-trial variant (20 trials × 3 sizes)
# ═══════════════════════════════════════════════════════════════════════════════

POOL_PATH           = "../POOL/composition_pool.csv"
A4_PATH             = "../../DATASET_A4/CombinedDataset_A4.csv"
T_REF               = 220.0   # K — fixed reference temperature

N_SIZES             = [25, 50, 75, 100]          # sample sizes (no N075)
N_MAX               = max(N_SIZES)           # 100
N_TRIALS            = 20
BASE_SEED           = 56852145               # trial i uses seed BASE_SEED + i

N_t                 = None
U_THRESHOLD         = 1e-6

GPR_MAX_SAMPLES     = 5000
RESTART_OPTIMIZER   = 3
GPR_SEED            = BASE_SEED

OUTPUT_DIR          = "AL_ST"
OUTPUT_BASE         = os.path.join(OUTPUT_DIR, "OUTPUT")   # AL_ST/OUTPUT/

for N in N_SIZES:
    os.makedirs(os.path.join(OUTPUT_BASE, f"N{N:03d}"), exist_ok=True)

print(f"N_SIZES    : {N_SIZES}")
print(f"N_TRIALS   : {N_TRIALS}")
print(f"BASE_SEED  : {BASE_SEED}")
print(f"T_REF      : {T_REF} K")
print(f"Output dir : {OUTPUT_BASE}/N0XX/trial_XX.csv")

N_SIZES    : [25, 50, 75, 100]
N_TRIALS   : 20
BASE_SEED  : 56852145
T_REF      : 220.0 K
Output dir : AL_ST/OUTPUT/N0XX/trial_XX.csv


In [3]:
# ── Column-name bridge ────────────────────────────────────────────────────────
CSV_TO_FULL = {
    "CO2" : "carbon dioxide",
    "H2"  : "hydrogen",
    "Ar"  : "argon",
    "N2"  : "nitrogen",
    "CH4" : "methane",
    "O2"  : "oxygen",
    "CO"  : "carbon monoxide",
    "H2S" : "hydrogen sulfide",
}

POOL_Z_COLS = list(CSV_TO_FULL.keys())
Z_FEATURES  = [f"z_{v}" for v in CSV_TO_FULL.values()]
COMP_MAP    = {k: f"z_{v}" for k, v in CSV_TO_FULL.items()}
FEATURES    = Z_FEATURES   # 8 features — composition only, no temperature
TARGET      = "P_bubble"

print(f"GPR features : {FEATURES}  ({len(FEATURES)} total)")

GPR features : ['z_carbon dioxide', 'z_hydrogen', 'z_argon', 'z_nitrogen', 'z_methane', 'z_oxygen', 'z_carbon monoxide', 'z_hydrogen sulfide']  (8 total)


In [4]:
# ── Load candidate pool ───────────────────────────────────────────────────────
pool = pd.read_csv(POOL_PATH)
print(f"Pool: {len(pool):,} compositions")

Pool: 1,000 compositions


In [5]:
# ── Load A4 at T_REF, fit GPR — ONE fit shared by all 20 trials ───────────────
a2         = pd.read_csv(A4_PATH)
a2.columns = [c.strip() for c in a2.columns]

extra_z = [c for c in a2.columns if c.startswith("z_") and c not in Z_FEATURES]
if extra_z:
    a2 = a2[a2[extra_z].sum(axis=1) < 1e-6].copy()
    print(f"8-component rows: {len(a2):,}")

a2_ref = a2[np.abs(a2["temperature"] - T_REF) <= 1.0].copy()
if len(a2_ref) < 100:
    print(f"Only {len(a2_ref)} rows at T_REF — using all temperatures")
    a2_ref = a2.copy()

df_train = (
    a2_ref[Z_FEATURES + [TARGET]]
    .drop_duplicates(subset=Z_FEATURES)
    .dropna(subset=[TARGET])
    .reset_index(drop=True)
)
if len(df_train) > GPR_MAX_SAMPLES:
    df_train = df_train.sample(n=GPR_MAX_SAMPLES, random_state=GPR_SEED).reset_index(drop=True)

print(f"GPR training set : {len(df_train):,} rows  (T_REF={T_REF} K)")
print(f"P_bubble range   : {df_train[TARGET].min():.2f} – {df_train[TARGET].max():.2f} bar")

n_feat = len(FEATURES)
kernel = (
    ConstantKernel(1.0, (1e-3, 1e3))
    * Matern(nu=2.5, length_scale=np.ones(n_feat), length_scale_bounds=(1e-3, 1e3))
    + WhiteKernel(noise_level=0.1, noise_level_bounds=(1e-4, 100.0))
)
gpr_model = Pipeline([
    ("scaler", StandardScaler()),
    ("gpr",    GaussianProcessRegressor(
        kernel               = kernel,
        alpha                = 0.0,
        normalize_y          = True,
        n_restarts_optimizer = RESTART_OPTIMIZER,
        random_state         = GPR_SEED,
    )),
])

t0 = time.perf_counter()
gpr_model.fit(df_train[FEATURES], df_train[TARGET])
print(f"GPR fitting time : {time.perf_counter()-t0:.1f} s")

GPR training set : 99 rows  (T_REF=220.0 K)
P_bubble range   : 7.22 – 109.51 bar
GPR fitting time : 0.6 s


/home/darshan/A6/py_A6/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/darshan/A6/py_A6/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 5 of parameter k1__k2__length_scale is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/darshan/A6/py_A6/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 6 of parameter k1__k2__length_scale is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/darshan/A6/py_A6/lib/python3.12/site-pac

In [6]:
# ── Extract signal kernel for Algorithm 1 ─────────────────────────────────────
gpr_step      = gpr_model.named_steps["gpr"]
fitted_k      = gpr_step.kernel_
signal_kernel = fitted_k.k1

ls  = signal_kernel.k2.length_scale
imp = 1.0 / ls;  imp /= imp.sum()
print("ARD feature importances:")
for name, val in sorted(zip(FEATURES, imp), key=lambda x: -x[1]):
    print(f"  {name:<30s}  {val*100:5.1f}%")

ARD feature importances:
  z_hydrogen                       53.9%
  z_hydrogen sulfide               16.1%
  z_carbon dioxide                 12.6%
  z_methane                         6.3%
  z_nitrogen                        5.3%
  z_argon                           1.9%
  z_oxygen                          1.9%
  z_carbon monoxide                 1.9%


In [7]:
# ── Scale pool using fitted StandardScaler ────────────────────────────────────
scaler       = gpr_model.named_steps["scaler"]
pool_renamed = pool.rename(columns=COMP_MAP)
X_pool       = scaler.transform(pool_renamed[FEATURES].values)   # (1000, 8)
print(f"Pool feature matrix shape : {X_pool.shape}")

Pool feature matrix shape : (1000, 8)


/home/darshan/A6/py_A6/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [8]:
# ── Algorithm 1 — kernel-based active learning ────────────────────────────────

def compute_U_T(X_T, X_S, kernel):
    K_TT_diag = kernel.diag(X_T)
    K_TS      = kernel(X_T, X_S)
    K_SS      = kernel(X_S, X_S)
    K_SS_reg  = K_SS + 1e-8 * np.eye(len(X_S))
    try:
        L = np.linalg.cholesky(K_SS_reg)
        V = np.linalg.solve(L, K_TS.T)
        U = K_TT_diag - np.einsum("ij,ij->j", V, V)
    except np.linalg.LinAlgError:
        K_inv = np.linalg.pinv(K_SS_reg)
        U = K_TT_diag - np.einsum("ij,jk,ki->i", K_TS, K_inv, K_TS.T)
    return np.maximum(U, 0.0)


def run_algorithm1(X_all, kernel, n_max, n_t=None, u_t=1e-6, seed=0):
    rng    = np.random.default_rng(seed)
    n_pool = len(X_all)
    init   = rng.choice(n_pool, size=2, replace=False)
    s_list = list(init)
    p_set  = set(range(n_pool)) - set(s_list)

    while p_set and len(s_list) < n_max:
        p_list = list(p_set)
        if n_t is not None and len(p_list) > n_t:
            t_list = [p_list[i] for i in rng.choice(len(p_list), size=n_t, replace=False)]
        else:
            t_list = p_list

        U          = compute_U_T(X_all[t_list], X_all[s_list], kernel)
        best_local = int(np.argmax(U))
        if U[best_local] > u_t:
            s_list.append(t_list[best_local])
            p_set.discard(t_list[best_local])
        for i, idx in enumerate(t_list):
            if U[i] < u_t:
                p_set.discard(idx)
    return s_list


print("Algorithm 1 defined.")

Algorithm 1 defined.


In [9]:
# ── P_bubble helper ───────────────────────────────────────────────────────────
ALL_COMPS_FULL = [CSV_TO_FULL[k] for k in POOL_Z_COLS]

def compute_pbubble_single(z_arr, T_K):
    active_z, active_comps, _ = CompositionHandler.reduce_components(
        z_arr, ALL_COMPS_FULL, verbose=False)
    params  = ParameterBuilder.build_parameters(active_comps)
    func    = feos.HelmholtzEnergyFunctional.pcsaft(params)
    feed    = CompositionHandler.compute_feed_moles(active_z)
    _, P_bub = VLECalculator.compute_bubble_curve(func, [T_K], feed, verbose=False)
    return float(P_bub[0]) if len(P_bub) else float("nan")


def make_selection_df(idx_list, pool_df, pbubble_cache):
    rows = pool_df.iloc[idx_list].copy().reset_index(drop=True)
    rows["pool_idx"] = idx_list
    rows["P_bubble"] = [pbubble_cache.get(i, float("nan")) for i in idx_list]
    return rows


print("P_bubble helper defined.")

P_bubble helper defined.


In [10]:
# ── 20-trial loop ─────────────────────────────────────────────────────────────
# The GPR kernel is fixed (fitted once at T_REF above).
# Each trial varies only the Algorithm 1 initialisation seed.

t_loop = time.perf_counter()
print(f"Generating {N_TRIALS} trials x {len(N_SIZES)} sizes at T_REF={T_REF} K …")
print(f"Output: {OUTPUT_BASE}/N0XX/trial_XX.csv")
print()

for trial in range(N_TRIALS):
    seed = BASE_SEED + trial

    # --- Algorithm 1 ---
    al_order = run_algorithm1(
        X_all  = X_pool,
        kernel = signal_kernel,
        n_max  = N_MAX,
        n_t    = N_t,
        u_t    = U_THRESHOLD,
        seed   = seed,
    )

    # --- Build snapshots at each N ---
    al_selections = {}
    for N in N_SIZES:
        n_avail = min(N, len(al_order))
        chosen  = list(al_order[:n_avail])
        if n_avail < N:
            fallback = [i for i in range(len(pool)) if i not in set(chosen)]
            rng_fb   = np.random.default_rng(seed + N)
            rng_fb.shuffle(fallback)
            chosen = chosen + fallback[:N - n_avail]
        al_selections[N] = chosen

    # --- Unique pool indices for this trial ---
    needed_idx = sorted(set(idx for sel in al_selections.values() for idx in sel))

    # --- Compute P_bubble at T_REF ---
    pbubble_cache = {}
    n_failed = 0
    for idx in needed_idx:
        row   = pool.iloc[idx]
        z_arr = np.array([row[k] for k in POOL_Z_COLS])
        try:
            p = compute_pbubble_single(z_arr, T_REF)
        except Exception:
            p = float("nan")
            n_failed += 1
        pbubble_cache[idx] = p

    n_ok = sum(1 for v in pbubble_cache.values() if not np.isnan(v))

    # --- Assemble and save ---
    for N in N_SIZES:
        df    = make_selection_df(al_selections[N], pool, pbubble_cache)
        fpath = os.path.join(OUTPUT_BASE, f"N{N:03d}", f"trial_{trial:02d}.csv")
        df.to_csv(fpath, index=False)

    print(f"  Trial {trial:02d} (seed={seed}):  {len(al_order):3d} selected  "
          f"{n_ok}/{len(needed_idx)} P_bubble OK")

elapsed_loop = time.perf_counter() - t_loop
total_files  = N_TRIALS * len(N_SIZES)
print()
print(f"Done — {total_files} CSVs saved in {elapsed_loop:.1f} s")
print(f"Total notebook time : {(time.perf_counter() - notebook_start)/60:.1f} min")

Generating 20 trials x 4 sizes at T_REF=220.0 K …
Output: AL_ST/OUTPUT/N0XX/trial_XX.csv

  Trial 00 (seed=56852145):   87 selected  100/100 P_bubble OK
  Trial 01 (seed=56852146):   86 selected  100/100 P_bubble OK
  Trial 02 (seed=56852147):   86 selected  100/100 P_bubble OK
  Trial 03 (seed=56852148):   88 selected  100/100 P_bubble OK
  Trial 04 (seed=56852149):   88 selected  100/100 P_bubble OK
  Trial 05 (seed=56852150):   90 selected  100/100 P_bubble OK
  Trial 06 (seed=56852151):   87 selected  100/100 P_bubble OK
  Trial 07 (seed=56852152):   86 selected  100/100 P_bubble OK
  Trial 08 (seed=56852153):   86 selected  100/100 P_bubble OK
  Trial 09 (seed=56852154):   87 selected  100/100 P_bubble OK
  Trial 10 (seed=56852155):   90 selected  100/100 P_bubble OK
  Trial 11 (seed=56852156):   87 selected  100/100 P_bubble OK
  Trial 12 (seed=56852157):   87 selected  100/100 P_bubble OK
  Trial 13 (seed=56852158):   88 selected  100/100 P_bubble OK
  Trial 14 (seed=56852159): 